# Live Dashboard - fno / dmf
Run this notebook in a customer workspace to execute the folder's KQL tiles, generate local dashboard output, and capture observations.
Each folder includes a committed dry-run `live-output/index.html` preview. Real customer runs should write to `live-dashboard-output/`, which is ignored by Git.


In [ ]:
from pathlib import Path
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'tools' / 'live_dashboard_runner.py').exists():
            return candidate
    raise RuntimeError('Could not find repository root containing tools/live_dashboard_runner.py')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
MANIFEST = Path.cwd() / 'LIVE-DASHBOARD.json'
OUTPUT_DIR = Path.cwd() / 'live-dashboard-output'


## Configure
Set `DRY_RUN = False` and provide either `APPINSIGHTS_RESOURCE_ID`, `WORKSPACE_ID`, or `SUBSCRIPTIONS` depending on the manifest runtime.


In [ ]:
DRY_RUN = True
APPINSIGHTS_RESOURCE_ID = ''  # /subscriptions/<sub>/resourceGroups/<rg>/providers/microsoft.insights/components/<name>
WORKSPACE_ID = ''             # Optional Log Analytics workspace ID
SUBSCRIPTIONS = ''           # Comma-separated subscription IDs for Azure Resource Graph manifests
TIMESPAN_DAYS = 30


In [ ]:
cmd = [
    sys.executable, str(REPO_ROOT / 'tools' / 'live_dashboard_runner.py'),
    '--manifest', str(MANIFEST),
    '--output', str(OUTPUT_DIR),
    '--timespan-days', str(TIMESPAN_DAYS),
]
if DRY_RUN:
    cmd.append('--dry-run')
if APPINSIGHTS_RESOURCE_ID:
    cmd.extend(['--appinsights-resource-id', APPINSIGHTS_RESOURCE_ID])
if WORKSPACE_ID:
    cmd.extend(['--workspace-id', WORKSPACE_ID])
if SUBSCRIPTIONS:
    cmd.extend(['--subscriptions', SUBSCRIPTIONS])
print('Running:', ' '.join(cmd))
completed = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
if completed.returncode != 0:
    raise SystemExit(completed.returncode)


## Review Output
Open `live-dashboard-output/index.html` for the rendered dashboard and `live-dashboard-output/observations.md` for the observation log.
The committed `live-output/index.html` file is only a dry-run preview for this folder.
Re-run with `DRY_RUN = False` after confirming credentials and RBAC.
